# 11 — A national transport model: Great Britain

This notebook builds and runs a **national strategic model of Great Britain**
from scratch. **Model size: 64,733 links (~40,000 km of motorways, trunk roads
and ramps), ~46,000 trigger-built nodes, and 88 zones** from the 75 largest
cities and strategic towns, with megacities split into sector zones. Gravity
demand is calibrated to plausible strategic-traffic levels and a national
equilibrium assignment finishes with separate flow and congestion map views,
the busiest corridors in the country, and the England–Scotland border screenline.

Everything the notebook needs ships in the `data/` folder next to it:

| file | contents | source |
|---|---|---|
| `uk_strategic_roads.geojson.gz` | 64,733 links / ~40,000 km of GB motorways, trunk roads and ramps, topologically connected | © OpenStreetMap contributors (ODbL), extracted via Overpass API |
| `uk_cities.csv` | 75 cities and strategic towns with coordinates and population | assembled from public population estimates |
| `gb_boundary.geojson` | simplified GB outline for the maps | public-domain world boundaries |

No downloads happen at run time — the model is fully self-contained.

In [1]:
import gzip
import json
import time
import warnings
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import LineString, shape

warnings.filterwarnings("ignore")

DATA = Path("data")
with gzip.open(DATA / "uk_strategic_roads.geojson.gz", "rt", encoding="utf-8") as fh:
    roads_gj = json.load(fh)
cities = pd.read_csv(DATA / "uk_cities.csv")
boundary = gpd.GeoDataFrame(geometry=[shape(f["geometry"]) for f in json.load(open(DATA / "gb_boundary.geojson"))["features"]], crs=4326)

roads = gpd.GeoDataFrame(
    [{"cls": f["properties"]["class"], "ref": f["properties"]["ref"]} for f in roads_gj["features"]],
    geometry=[shape(f["geometry"]) for f in roads_gj["features"]], crs=4326)

km = roads.to_crs(27700).length.sum() / 1000
print(f"{len(roads):,} links / {km:,.0f} km; {len(cities)} cities, "
      f"{cities.population.sum() / 1e6:.1f}M residents")
roads.groupby("cls").size()

64,733 links / 37,206 km; 74 cities, 34.7M residents


cls
motorway     3172
ramp        13862
trunk       47699
dtype: int64

In [2]:
# Offline map helper ---------------------------------------------------------
# Interactive maps with no server extensions, no labextensions beyond the
# ipywidgets manager, and no CDN: lonboard renders WebGL maps whose frontend
# JavaScript ships from the kernel through the ipywidgets channel.
#
# Backends (AEQ_MAP_BACKEND environment variable):
#   lonboard (default) - interactive WebGL maps (pip install lonboard anywidget)
#   static             - matplotlib rendering, works absolutely anywhere
#
# The declarative symbology below (field()/constant() chains) is self-contained
# and renders identically on both backends.
import os

import matplotlib.colors
import matplotlib.pyplot as _plt
import numpy as np


# --- declarative symbology --------------------------------------------------
class _Mapping:
    def __init__(self, field, scheme, params):
        self.field, self.scheme, self.params = field, scheme, params

    def encoding(self, *targets):
        return {"field": self.field, "scheme": self.scheme,
                "params": self.params, "encodings": list(targets)}


class _Field:
    def __init__(self, name):
        self.name = name

    def colormap(self, name="viridis", *, domain=None, reverse=False, n_shades=9):
        return _Mapping(self.name, "colormap",
                        {"name": name, "domain": domain, "reverse": reverse})

    def scalar(self, *, domain, output_range):
        return _Mapping(self.name, "scalar",
                        {"domain": list(domain), "range": list(output_range)})

    def categorical(self, name="tab10"):
        return _Mapping(self.name, "categorical", {"name": name})


class _Constant:
    def __init__(self, value):
        self.value = value

    def encoding(self, *targets):
        scheme = "constant_num" if isinstance(self.value, (int, float)) else "constant_color"
        return {"field": None, "scheme": scheme,
                "params": {"value": self.value}, "encodings": list(targets)}


def field(name):
    """Style by a data column: .colormap() / .scalar() / .categorical()."""
    return _Field(name)


def constant(value):
    """A fixed colour (hex/name) or number, e.g. constant("#dc2626")."""
    return _Constant(value)


def _rgba255(c, alpha=1.0):
    r, g, b, a = matplotlib.colors.to_rgba(c, alpha)
    return [int(r * 255), int(g * 255), int(b * 255), int(a * 255)]


def _style_arrays(symbology, gdf):
    """symbology -> per-row uint8 RGBA arrays and float width arrays."""
    n = len(gdf)
    out = {"stroke": None, "width": None, "fill": None}
    if not symbology:
        return out
    mappings = [m for group in symbology for m in (group if isinstance(group, list) else [group])]
    for m in mappings:
        scheme, params, fld, encs = m["scheme"], m["params"], m["field"], m["encodings"]
        arr = wid = None
        if scheme == "constant_color":
            arr = np.tile(_rgba255(params["value"]), (n, 1)).astype(np.uint8)
        elif scheme == "colormap":
            cmap = _plt.get_cmap(params["name"])
            if params.get("reverse"):
                cmap = cmap.reversed()
            dom = params.get("domain") or [float(gdf[fld].min()), float(gdf[fld].max())]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - dom[0]) / max(dom[1] - dom[0], 1e-12), 0, 1)
            rgba = cmap(t)
            arr = (rgba * 255).astype(np.uint8)
        elif scheme == "categorical":
            cmap = _plt.get_cmap(params["name"])
            uniq = list(dict.fromkeys(gdf[fld].dropna()))
            idx = {v: i for i, v in enumerate(uniq)}
            arr = np.array([_rgba255(cmap(idx.get(v, 0) % cmap.N)) for v in gdf[fld]], dtype=np.uint8)
        elif scheme == "constant_num":
            wid = np.full(n, float(params["value"]))
        elif scheme == "scalar":
            d, r = params["domain"], params["range"]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - d[0]) / max(d[1] - d[0], 1e-12), 0, 1)
            wid = r[0] + t * (r[1] - r[0])
        if arr is not None:
            if any("stroke" in e for e in encs):
                out["stroke"] = arr
            if any("fill" in e for e in encs):
                out["fill"] = arr
        if wid is not None and any("width" in e for e in encs):
            out["width"] = wid
    return out


# --- the map document -------------------------------------------------------
class MapDoc:
    """Collects styled layers; displays via lonboard (WebGL) or matplotlib."""

    def __init__(self):
        self.items = []  # (gdf, name, arrays, opacity)

    def add(self, gdf, name, symbology, opacity):
        g = gdf.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True)
        self.items.append((g, name, _style_arrays(symbology, g), opacity))

    def _lonboard_map(self):
        from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
        layers = []
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            base = g[["geometry"]]
            if "LineString" in geom:
                kw = {"width_units": "pixels", "width_min_pixels": 1.0, "opacity": op}
                if st["stroke"] is not None:
                    kw["get_color"] = st["stroke"]
                if st["width"] is not None:
                    kw["get_width"] = st["width"]
                layers.append(PathLayer.from_geopandas(base, **kw))
            elif "Polygon" in geom:
                kw = {"opacity": op * 0.6, "stroked": False}
                if st["fill"] is not None:
                    kw["get_fill_color"] = st["fill"]
                layers.append(PolygonLayer.from_geopandas(base, **kw))
            else:
                kw = {"radius_min_pixels": 5, "opacity": op}
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                if fill is not None:
                    kw["get_fill_color"] = fill
                layers.append(ScatterplotLayer.from_geopandas(base, **kw))
        return Map(layers=layers, basemap=None)

    def _static_figure(self):
        fig, ax = _plt.subplots(figsize=(9, 7))
        ax.set_facecolor("#eef1f4")
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            if "LineString" in geom:
                colors = st["stroke"] / 255 if st["stroke"] is not None else "#1d4ed8"
                widths = st["width"] if st["width"] is not None else 1.0
                g.plot(ax=ax, color=colors, linewidth=widths, alpha=op)
            elif "Polygon" in geom:
                colors = st["fill"] / 255 if st["fill"] is not None else "#cbd5e1"
                g.plot(ax=ax, color=colors, alpha=op * 0.6)
            else:
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                g.plot(ax=ax, color=(fill / 255 if fill is not None else "#dc2626"),
                       markersize=25, alpha=op)
        ax.set_aspect(1.4)
        ax.set_xticks([]), ax.set_yticks([])
        _plt.tight_layout()
        _plt.close(fig)
        return fig

    def _ipython_display_(self):
        from IPython.display import display
        be = os.environ.get("AEQ_MAP_BACKEND", "lonboard").strip().lower()
        display(self._static_figure() if be == "static" else self._lonboard_map())


def new_map(gdf_for_extent=None, zoom=12):
    """Create a map document (extent/zoom args kept for API compatibility;
    lonboard auto-fits to its layers)."""
    return MapDoc()


def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a styled layer."""
    doc.add(gdf, name, symbology, kwargs.get("opacity", 1.0))
    return name


def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature — backdrop
    layers do not need per-feature identity, and one merged feature is a
    fraction of the size and draw cost."""
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)


## The raw ingredients

The strategic road network, the GB outline and the city zones.

In [3]:
# field()/constant() symbology builders come from the map helper cell

cities_gdf = gpd.GeoDataFrame(cities, geometry=gpd.points_from_xy(cities.lon, cities.lat), crs=4326)

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, merge_lines(roads[roads.cls == "trunk"]), "trunk roads",
        symbology=[[constant("#64748b").encoding("stroke")]])
add_gdf(doc, merge_lines(roads[roads.cls == "motorway"]), "motorways",
        symbology=[[constant("#1d4ed8").encoding("stroke")]])
add_gdf(doc, cities_gdf[["city", "population", "geometry"]], "cities",
        symbology=[[constant("#dc2626").encoding("fill")]])
doc

[interactive offline map - run the notebook to display]

## 1. Building the national network

Every link is inserted through AequilibraE's standard editing path, so the
database consistency triggers create the nodes and maintain `a_node`/`b_node`
and lengths. Class-based speeds and per-direction capacities give the free-flow
attributes.

In [4]:
from aequilibrae.project import Project

SPEC = {  # free-flow speed km/h, capacity veh/h/direction
    "motorway": (105, 4000),
    "trunk": (75, 1800),
    "ramp": (55, 1500),
}

fldr = str(Path(gettempdir()) / uuid4().hex)
project = Project()
project.new(fldr)

t0 = time.perf_counter()
with project.db_connection as conn:
    for lt, (speed, cap) in SPEC.items():
        conn.execute("insert into link_types (link_type, link_type_id, description, speed) values (?,?,?,?)",
                     (lt, lt[0], f"{lt} (UK network)", speed))
    lid = 0
    for f in roads_gj["features"]:
        cls = f["properties"]["class"]
        coords = f["geometry"]["coordinates"]
        if coords[0] == coords[-1]:
            continue  # degenerate rings cannot carry through traffic
        speed, cap = SPEC[cls]
        lid += 1
        wkt = "LINESTRING(" + ", ".join(f"{x} {y}" for x, y in coords) + ")"
        conn.execute(
            "insert into links (link_id, a_node, b_node, link_type, modes, direction, "
            " speed_ab, speed_ba, capacity_ab, capacity_ba, name, geometry) "
            "values (?, 0, 0, ?, 'c', 0, ?, ?, ?, ?, ?, GeomFromText(?, 4326))",
            (lid, cls, speed, speed, cap, cap, f["properties"]["ref"], wkt))
    conn.execute("update links set travel_time_ab = distance / 1000.0 / speed_ab * 60, "
                 "travel_time_ba = distance / 1000.0 / speed_ba * 60")
    conn.commit()
    n_nodes = conn.execute("select count(*) from nodes").fetchone()[0]
print(f"inserted {lid:,} links in {time.perf_counter() - t0:.0f}s; "
      f"triggers created {n_nodes:,} nodes")

inserted 64,730 links in 54s; triggers created 46,438 nodes


## 2. Cities become zones

Point zones are too coarse for a 9-million-resident metropolis: all its traffic
would funnel through a couple of streets. Cities over 1M residents are split
into a centre plus a ring of sector zones, and every zone is tied into the
network by several spread-out centroid connectors.

In [5]:
from scipy.spatial import cKDTree

# Megacities cannot be a single point: their trips start all over the metro area,
# so cities over 1M residents are split into a centre plus ring of sector zones
# (standard practice in strategic models), each with its share of the population.
subs = []
for c in cities.itertuples():
    k = int(np.clip(round(c.population / 1_500_000) + 1, 1, 6)) if c.population >= 1_000_000 else 1
    if k == 1:
        subs.append(dict(zone=c.city, parent=c.city, lat=c.lat, lon=c.lon,
                         population=c.population, core=True))
    else:
        r = 0.10 if c.population > 5e6 else 0.05
        subs.append(dict(zone=f"{c.city} C", parent=c.city, lat=c.lat, lon=c.lon,
                         population=c.population / k, core=True))
        for i in range(k - 1):
            ang = 2 * np.pi * i / (k - 1)
            subs.append(dict(zone=f"{c.city} S{i+1}", parent=c.city,
                             lat=c.lat + r * 0.7 * np.sin(ang),
                             lon=c.lon + r * np.cos(ang) / np.cos(np.radians(54.5)),
                             population=c.population / k, core=False))
zones_df = pd.DataFrame(subs)

# Each zone gets 2-5 connectors (more for bigger zones), spread over distinct
# entry points so demand does not funnel through a single street.
nodes_df = project.network.nodes.data
xy = np.c_[nodes_df.geometry.x * np.cos(np.radians(54.5)), nodes_df.geometry.y]
tree = cKDTree(xy)

with project.db_connection as conn:
    for z in zones_df.itertuples():
        n_conn = int(np.clip(2 + z.population / 500_000, 2, 5))
        _, idx = tree.query([z.lon * np.cos(np.radians(54.5)), z.lat], k=200)
        chosen = []
        for j in np.atleast_1d(idx):
            pt = nodes_df.geometry.iloc[int(j)]
            if all(abs(pt.x - q.x) + abs(pt.y - q.y) > 0.02 for q in chosen):
                chosen.append(pt)
            if len(chosen) == n_conn:
                break
        for pt in chosen:
            lid += 1
            conn.execute(
                "insert into links (link_id, a_node, b_node, link_type, modes, direction, "
                " speed_ab, speed_ba, capacity_ab, capacity_ba, name, geometry) "
                "values (?, 0, 0, 'centroid_connector', 'c', 0, 48, 48, 10000, 10000, ?, GeomFromText(?, 4326))",
                (lid, f"{z.zone} connector", f"LINESTRING({z.lon} {z.lat}, {pt.x} {pt.y})"))
    conn.execute("update links set travel_time_ab = distance / 1000.0 / speed_ab * 60, "
                 "travel_time_ba = distance / 1000.0 / speed_ba * 60 where travel_time_ab is null")
    conn.commit()

nodes_df = project.network.nodes.data  # refresh: zone nodes now exist
key = (nodes_df.geometry.x.round(5).astype(str) + "|" + nodes_df.geometry.y.round(5).astype(str))
lookup = dict(zip(key, nodes_df.node_id))
zones_df["node_id"] = [lookup[f"{round(z.lon, 5)}|{round(z.lat, 5)}"] for z in zones_df.itertuples()]

with project.db_connection as conn:
    conn.executemany("update nodes set is_centroid = 1 where node_id = ?",
                     [(int(n),) for n in zones_df.node_id])
    conn.commit()
print(f"{len(zones_df)} zones from {cities.city.nunique()} cities "
      f"({(zones_df.groupby('parent').size() > 1).sum()} megacities sectored)")

88 zones from 74 cities (8 megacities sectored)


## 3. National skims

Free-flow drive times between all zones.

In [6]:
from aequilibrae.paths import NetworkSkimming

project.network.build_graphs(modes=["c"])
graph = project.network.graphs["c"]
graph.set_graph("travel_time")
graph.set_skimming(["travel_time", "distance"])
graph.set_blocked_centroid_flows(True)

t0 = time.perf_counter()
skimmer = NetworkSkimming(graph)
skimmer.execute()
tt = np.array(skimmer.results.skims.get_matrix("travel_time"), copy=True)
print(f"skimmed {tt.shape[0]}x{tt.shape[1]} zone pairs in {time.perf_counter() - t0:.1f}s")

by_node = zones_df.set_index("node_id").reindex(graph.centroids)
times = pd.DataFrame(tt, index=by_node.zone, columns=by_node.zone)
rep = dict(zip(zones_df[zones_df.core].parent, zones_df[zones_df.core].zone))
pairs = [("London", "Birmingham"), ("London", "Manchester"), ("London", "Edinburgh"),
         ("Manchester", "Glasgow"), ("Bristol", "Newcastle upon Tyne"), ("Cardiff", "Norwich")]
pd.DataFrame([{"from": a, "to": b, "free-flow drive": f"{times.loc[rep[a], rep[b]] / 60:.1f} h"}
              for a, b in pairs])

[interactive offline map - run the notebook to display]

skimmed 88x88 zone pairs in 0.2s


,from,to,free-flow drive
0,London,Birmingham,1.9 h
1,London,Manchester,3.2 h
2,London,Edinburgh,6.5 h
3,Manchester,Glasgow,3.4 h
4,Bristol,Newcastle upon Tyne,4.7 h
5,Cardiff,Norwich,4.7 h


## 4. National demand

Gravity demand between cities: productions from population, attractions from a
sublinear population proxy, a slow exponential decay suited to long-distance
travel, balanced with doubly-constrained IPF. Within-city movements are masked
out — they belong to urban models, not a strategic one.

The trip rate (0.4% of residents starting a strategic car trip in the peak hour)
is calibrated so the busiest corridors reach the edge of capacity while the
border screenline and the London–Birmingham movement land at plausible
magnitudes — the standard sanity anchors for a sketch national model.

In [7]:
pop = by_node.population.to_numpy(dtype=float)
parents = by_node.parent.to_numpy()

TRIP_RATE = 0.004   # ~0.4% of residents starting a strategic inter-city car trip in the peak hour
productions = pop * TRIP_RATE
attractions = np.power(pop, 0.9)
attractions *= productions.sum() / attractions.sum()

imp = tt.copy()
np.fill_diagonal(imp, np.nan)
imp[~np.isfinite(imp)] = np.nan

T = np.outer(productions, attractions) * np.exp(-0.02 * np.nan_to_num(imp, nan=1e4))
T[np.isnan(imp)] = 0.0
np.fill_diagonal(T, 0.0)
T[parents[:, None] == parents[None, :]] = 0.0   # within-city travel is not strategic demand
for it in range(100):
    rs = T.sum(1); T *= np.divide(productions, rs, out=np.zeros_like(rs), where=rs > 0)[:, None]
    cs = T.sum(0); T *= np.divide(attractions, cs, out=np.zeros_like(cs), where=cs > 0)[None, :]
    if np.abs(T.sum(1) - productions).sum() / productions.sum() < 1e-5:
        break

lon_bir = T[np.ix_(parents == "London", parents == "Birmingham")].sum()
print(f"{T.sum():,.0f} strategic trips in the peak hour; "
      f"London -> Birmingham {lon_bir:,.0f} veh/h")

138,885 strategic trips in the peak hour; London -> Birmingham 5,698 veh/h


## 5. National equilibrium assignment

In [8]:
from aequilibrae.matrix import AequilibraeMatrix
from aequilibrae.paths import TrafficAssignment, TrafficClass

demand = AequilibraeMatrix()
demand.create_empty(zones=graph.num_zones, matrix_names=["matrix"], memory_only=True)
demand.index = graph.centroids[:]
demand.matrices[:, :, 0] = T
demand.computational_view()

assig = TrafficAssignment()
assig.add_class(TrafficClass(name="car", graph=graph, matrix=demand))
assig.set_vdf("BPR")
assig.set_vdf_parameters({"alpha": 0.15, "beta": 4.0})
assig.set_capacity_field("capacity")
assig.set_time_field("travel_time")
assig.set_algorithm("bfw")
assig.max_iter = 30
assig.rgap_target = 0.001
t0 = time.perf_counter()
assig.execute()
print(f"equilibrium in {time.perf_counter() - t0:.0f}s")

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

equilibrium in 5s


## 6. Two views of the result

The same assignment, two questions: **where is the traffic?** (flow view — line
width and colour scale with vehicles/hour) and **where does the network
struggle?** (congestion view — colour is the volume/capacity ratio, green to
red, width still shows flow so busy-and-congested links dominate).

In [9]:
# field()/constant() symbology builders come from the map helper cell

res = assig.results()
links_gdf = project.network.links.data
loaded = links_gdf.merge(res.reset_index(), on="link_id")
loaded = loaded[(loaded.matrix_tot > 100) & (loaded.link_type != "centroid_connector")].copy()
loaded["flow"] = loaded.matrix_tot.round(0)
loaded["voc"] = loaded["VOC_max"].clip(upper=1.5).round(3)
loaded["geometry"] = loaded.geometry.simplify(0.005)
fmax = float(loaded.flow.max())

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, loaded[["link_id", "flow", "geometry"]], "traffic flow (veh/h)",
        symbology=[[field("flow").colormap("YlOrRd", domain=(0.0, fmax)).encoding("stroke"),
                    field("flow").scalar(domain=(100.0, fmax), output_range=(0.6, 7.0)).encoding("stroke-width")]])
doc

[interactive offline map - run the notebook to display]

And the congestion view — the same flows coloured by volume/capacity:

In [10]:
used = loaded[loaded.flow > 0]
print(f"congestion: mean V/C {used.VOC_max.mean():.2f}, max {used.VOC_max.max():.2f}, "
      f"{(used.VOC_max > 1).mean() * 100:.1f}% of used links over capacity")

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, loaded[["link_id", "voc", "flow", "geometry"]], "congestion (V/C)",
        symbology=[[field("voc").colormap("RdYlGn", reverse=True, domain=(0.0, 1.5)).encoding("stroke"),
                    field("flow").scalar(domain=(100.0, fmax), output_range=(0.6, 7.0)).encoding("stroke-width")]])
doc

congestion: mean V/C 0.28, max 2.47, 3.7% of used links over capacity


[interactive offline map - run the notebook to display]

The busiest strategic corridors, ranked by vehicle-kilometres:

In [11]:
loaded["veh_km"] = loaded["matrix_tot"] * loaded["distance"] / 1000
corridors = (loaded[loaded["name"].str.len() > 0].groupby("name")
             .agg(veh_km=("veh_km", "sum"), peak_flow=("matrix_tot", "max"))
             .nlargest(10, "veh_km").round(0).astype(int))
corridors

,veh_km,peak_flow
name,,
M1,4147924,11967
M4,2442502,10782
M40,2158491,8554
M6,2150859,5958
A1(M),991607,3671
M62,857081,5522
M5,667295,2572
A74(M),460654,3249
M3,408556,2256


The same story as a map — each of the ten busiest corridors in its own colour, line width scaled by flow:

In [12]:
cor = loaded[loaded["name"].isin(corridors.index)]

doc = new_map(boundary, zoom=6)
add_gdf(doc, boundary, "Great Britain", opacity=0.2, symbology=[[constant("#94a3b8").encoding("fill")]])
add_gdf(doc, merge_lines(loaded), "strategic network", opacity=0.3,
        symbology=[[constant("#94a3b8").encoding("stroke")]])
add_gdf(doc, cor[["link_id", "name", "flow", "geometry"]], "busiest corridors",
        symbology=[[field("name").categorical("tab10").encoding("stroke"),
                    field("flow").scalar(domain=(100.0, fmax), output_range=(1.5, 7.0)).encoding("stroke-width")]])
doc

[interactive offline map - run the notebook to display]

## 7. The England–Scotland border screenline

In [13]:
border = LineString([(-3.6, 54.98), (-1.8, 55.82)])
crossing = loaded[loaded.geometry.intersects(border)]
print(f"{len(crossing)} links cross the border screenline; "
      f"{crossing.matrix_tot.sum():,.0f} vehicles/h in the modeled peak:")
crossing[["name", "link_type", "matrix_tot"]].sort_values("matrix_tot", ascending=False) \
        .rename(columns={"matrix_tot": "flow"}).round(0).reset_index(drop=True)

3 links cross the border screenline; 6,145 vehicles/h in the modeled peak:


,name,link_type,flow
0,A74(M),motorway,3038.0
1,A74(M),motorway,2767.0
2,A68,trunk,340.0


## Wrap-up, provenance and caveats

A national model, end to end: 64,000 real strategic-road links inserted through
the consistency triggers, ~90 population-weighted zones with megacity sectoring,
free-flow national skims, gravity + IPF demand calibrated to plausible
screenline levels, a converged BPR equilibrium, separate flow and congestion
views, corridor rankings and a border screenline — all from three small files
in `data/`. Notebook 12 rebuilds this model on a much finer network.

**Data provenance.** The road network is © OpenStreetMap contributors, licensed
ODbL, extracted from the Overpass API (motorway, trunk and their ramps; merged,
topologically noded, largest connected component) in August 2026. The GB outline
is from public-domain world boundary data. City populations are approximate
urban-area figures assembled from public estimates.

**Caveats.** This is a teaching model: the demand rates are calibrated only to
order-of-magnitude anchors, the network carries no local roads, and free-flow
times ignore junction delay. Remaining hotspots above capacity sit on inner-city
trunk streets, where a point-zone model concentrates traffic that in reality
disperses onto local roads.